# EDA

In [1]:
import pandas as pd
from ydata_profiling import ProfileReport
import warnings
warnings.filterwarnings('ignore')

df=pd.read_excel('../data/raw/04_cleaned_data.xlsx',index_col=0)

## Preliminary Processing

In [2]:
# remove original cols replaced by handled cols
columns_to_drop = [
    'battery_capacity',      # replaced by battery_capacity_avg
    'range_CLTC',            # replaced by range_CLTC_avg
    'price',                 # replaced by price_avg
    'battery_type',          # replaced by battery_type_clean
    'battery_supplier',      # replaced by battery_supplier_clean
    'radar_type',            # replaced by has_lidar
    'body_structure'         # replaced by seats
]

df = df.drop(columns=columns_to_drop)

In [3]:
# shift of numeric cols

numeric_columns = ['year','battery_capacity_avg', 'range_CLTC_avg', 'price_avg',
                   'length_mm', 'width_mm', 'depth_mm', 'wheel_base']

# Convert each to numeric (coerce errors to NaN)
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Convert year and seats to int (after numeric conversion)
df['seats']=df['seats'].astype('object')

In [4]:
print('=========================================')
print('basic info')
print('=========================================\n')
print('\n✅ shape\n')
print(df.shape)
print('\n✅ columns list\n')
print(df.columns.to_list())
print('\n✅ info statistic\n')
print(df.info())
print('\n✅ basic calculation\n')
print(df.describe())
print('\n✅ sample display\n')
print(df.head(5))

print('\n=========================================')
print('unique value of columns')
print('=========================================\n')
for col in df.columns:
    print(f'\n✅ {col}\n')
    print(df[f'{col}'].unique())

basic info


✅ shape

(243, 18)

✅ columns list

['year', 'brand_category', 'brand', 'model', 'segment', 'powertrain', 'length_mm', 'width_mm', 'depth_mm', 'wheel_base', 'ADAS_Level', 'battery_capacity_avg', 'range_CLTC_avg', 'price_avg', 'seats', 'battery_type_clean', 'battery_supplier_clean', 'has_lidar']

✅ info statistic

<class 'pandas.core.frame.DataFrame'>
Index: 243 entries, 62 to 117
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   year                    243 non-null    int64  
 1   brand_category          243 non-null    object 
 2   brand                   243 non-null    object 
 3   model                   243 non-null    object 
 4   segment                 243 non-null    object 
 5   powertrain              243 non-null    object 
 6   length_mm               243 non-null    int64  
 7   width_mm                242 non-null    float64
 8   depth_mm                243 non-null

## Auto EDA report

In [5]:
from ydata_profiling import ProfileReport

# Generate report without explorations parameter
profile = ProfileReport(df, 
                        title="Vehicle Pricing Analysis Report", 
                        sensitive=False,
                        correlations={
                            "pearson": {"calculate": True},
                            "spearman": {"calculate": True},
                            "cramers": {"calculate": True},
                        }
                       )

# Save report
profile.to_file("vehicle_data_profile.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|███████████████████████████████████████████| 18/18 [00:00<00:00, 23.13it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

## Further Observation

### Outlier

In [6]:
def count_outliers_all(df):
    numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
    results = []
    
    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        outlier_count = ((df[col] < lower) | (df[col] > upper)).sum()
        outlier_pct = outlier_count / len(df) * 100
        
        results.append({
            'Column': col,
            'Lower Bound': round(lower, 2),
            'Upper Bound': round(upper, 2),
            'Outliers Count': outlier_count,
            'Outliers %': f"{outlier_pct:.1f}%"
        })
    
    return pd.DataFrame(results)

# Run
outlier_df = count_outliers_all(df)
print(outlier_df.to_string(index=False))

              Column  Lower Bound  Upper Bound  Outliers Count Outliers %
                year      2017.25      2027.25               1       0.4%
           length_mm      4133.75      5471.75              24       9.9%
            width_mm      1693.12      2104.12              13       5.3%
            depth_mm      1276.50      1952.50               1       0.4%
          wheel_base      2402.00      3282.00              10       4.1%
battery_capacity_avg       -22.01       139.46               3       1.2%
      range_CLTC_avg      -347.50      1152.50               0       0.0%
           price_avg        -9.41        56.81              11       4.5%


### Imbalenced

In [7]:
def check_balance_all(df):
    categorical_cols = df.select_dtypes(include=['object']).columns
    results = []
    
    for col in categorical_cols:
        freq = df[col].value_counts(normalize=True)
        results.append({
            'Column': col,
            'Unique Categories': len(freq),
            'Minority Ratio': f"{freq.min():.2%}",
            'Categories <5%': (freq < 0.05).sum()
        })
    
    return pd.DataFrame(results)

# Run
balance_check = check_balance_all(df)
print(balance_check.to_string(index=False))

                Column  Unique Categories Minority Ratio  Categories <5%
        brand_category                  6          7.41%               0
                 brand                 76          0.41%              75
                 model                243          0.41%             243
               segment                 25          0.41%              18
            powertrain                  4          4.94%               1
            ADAS_Level                  3         18.52%               0
                 seats                  5          0.82%               2
    battery_type_clean                  5          0.82%               2
battery_supplier_clean                  3         26.75%               0
             has_lidar                  3         18.52%               0


### Correlations

In [8]:
# corr only detect correlations between numeric columns
# ============================================
print("\n" + "="*60)
print('correlations between features and target')
print("="*60)

target_col='price_avg'
feature_cols=[i for i in df.columns if i != target_col]

correlations = {}
for col in feature_cols:
    try:
        corr_val = df[col].corr(df[target_col])
        correlations[col] = corr_val
    except:
        pass
    
sorted_corr = pd.Series(correlations).sort_values(key=abs, ascending=False)

print("\nrank:")

for feat, corr_val in sorted_corr.items():
    strength = "strong" if abs(corr_val) > 0.5 else "medium" if abs(corr_val) > 0.3 else "weak"
    direction = "positive" if corr_val > 0 else "negative"
    print(f"  {feat}: {corr_val:.4f} ({direction}, {strength})")



correlations between features and target

rank:
  width_mm: 0.5580 (positive, strong)
  battery_capacity_avg: 0.5512 (positive, strong)
  wheel_base: 0.5376 (positive, strong)
  length_mm: 0.4998 (positive, medium)
  range_CLTC_avg: 0.3656 (positive, medium)
  depth_mm: 0.2038 (positive, weak)
  year: 0.0829 (positive, weak)


## File Saving

In [10]:
df.to_csv('../data/processed/final_before_feng.csv')